# KVO_06 v0.6 — U-Net-guided SAM3 Musanga crown extraction on Baego

**Notebook version: `KVO_06 v0.6`**  

**Question:** can the final KVO_04 Musanga probability surface efficiently identify candidate Musanga locations, after which SAM3 delineates individual crown objects?

This replaces expensive wall-to-wall SAM3 crown discovery. The division of labour is:

**KVO_04 U-Net → candidate Musanga probability peaks → local RGB chips → SAM3 crown delineation → object-level Musanga inventory**

`Crown_Nadine.shp` is used only for optional geometric QA where a candidate overlaps a high-confidence manually digitized crown. It is not required to generate candidates.

Outputs are candidate points, SAM3 crown polygons, per-object U-Net probability summaries, and QA tables. KVO_03 remains the independent species-classification validation; KVO_05 remains the dedicated SAM3 boundary benchmark.


## 0 — Host environment and configuration

Use a **GPU runtime**. KVO_06 reuses the same isolated official SAM3 setup as KVO_05.


In [ ]:
import sys, subprocess, os, json, random, warnings, gc, shutil, math
from pathlib import Path
print("Host Python:",sys.version)
subprocess.run(["nvidia-smi"],check=False)
!pip -q install -U "numpy>=2,<2.6" "huggingface_hub<2" geopandas rasterio pyogrio shapely scikit-learn matplotlib tqdm pillow


In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape, Point
from tqdm.auto import tqdm
from PIL import Image
from google.colab import drive
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED)
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT))

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"; GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_06_UNet_guided_SAM3"
CHIPS=OUT/"chips"; RAW=OUT/"sam3_raw"; VIS=OUT/"examples"
for p in [OUT,CHIPS,RAW,VIS]: p.mkdir(parents=True,exist_ok=True)

RASTER=GEO/"Baego_georef.tif"
PROB=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"/"Baego_musanga_probability_uint16.tif"
ANNOT=AD/"Crown_Nadine.shp"
assert RASTER.exists(),RASTER
assert PROB.exists(),PROB

# Candidate extraction. This is intentionally conservative and computationally cheap.
PROB_THRESHOLD=0.90
SCAN_WINDOW_PX=2048
PEAK_MIN_DISTANCE_M=10.0
CONTEXT_RADIUS_M=4.0       # score support around each peak
MIN_CONTEXT_MEAN=0.60      # reject isolated hot pixels
MAX_CANDIDATES=500         # first targeted SAM3 experiment
CHIP_SIZE_M=24.0          # local RGB context around each U-Net candidate
BOX_SIZE_M=10.0           # SAM3 geometric prompt centered on probability peak
MIN_PRED_AREA_M2=2.0
MAX_PRED_AREA_M2=500.0
FORCE_CHIPS=False
FORCE_SAM=False
print("OUT:",OUT)


## 1 — Extract coherent U-Net Musanga candidates and prepare RGB chips

The candidate detector now favors **spatially coherent high-probability regions**, not isolated hot pixels. It scans the KVO_04 Baego probability map in large windows, finds strong peaks, samples the mean probability in a local neighbourhood around each peak, rejects weak-context peaks, applies 10 m spatial suppression, then keeps the best 500 candidates ranked by a combined peak/context score.

This deliberately makes SAM3 a targeted crown delineator rather than an expensive wall-to-wall detector.


In [ ]:
# v0.5 — coherent candidate extraction.
# Pass 1: large-window scan; retain the strongest pixel per 10-m metric grid cell.
with rasterio.open(PROB) as ps:
    pcrs=ps.crs; ptr=ps.transform
    gsd=float((abs(ptr.a)+abs(ptr.e))/2)
    grid_m=float(PEAK_MIN_DISTANCE_M)
    nx=math.ceil(ps.width/SCAN_WINDOW_PX); ny=math.ceil(ps.height/SCAN_WINDOW_PX)
    nwin=nx*ny; best={}
    thr=int(round(PROB_THRESHOLD*10000))
    for r0 in tqdm(range(0,ps.height,SCAN_WINDOW_PX),total=ny,desc="U-NET SCAN"):
        for c0 in range(0,ps.width,SCAN_WINDOW_PX):
            hh=min(SCAN_WINDOW_PX,ps.height-r0); ww=min(SCAN_WINDOW_PX,ps.width-c0)
            a=ps.read(1,window=Window(c0,r0,ww,hh))
            rr,cc=np.where((a!=65535) & (a>=thr))
            if len(rr)==0: continue
            vals=a[rr,cc].astype(np.float32)/10000.0
            gr=rr+r0; gc=cc+c0
            xs,ys=rasterio.transform.xy(ptr,gr,gc,offset="center")
            xs=np.asarray(xs); ys=np.asarray(ys)
            gx=np.floor(xs/grid_m).astype(np.int64); gy=np.floor(ys/grid_m).astype(np.int64)
            order=np.argsort(vals)[::-1]; seen=set()
            for j in order:
                key=(int(gx[j]),int(gy[j]))
                if key in seen: continue
                seen.add(key)
                rec=(float(vals[j]),int(gr[j]),int(gc[j]),float(xs[j]),float(ys[j]))
                old=best.get(key)
                if old is None or rec[0]>old[0]: best[key]=rec

    print("Large raster windows scanned:",nwin)
    print("Strong coarse candidate cells:",len(best))

    # Pass 2: measure local support around each strong peak directly from probability map.
    rad=max(1,int(round(CONTEXT_RADIUS_M/gsd)))
    coherent=[]
    for peak,row,col,x,y in tqdm(best.values(),desc="CONTEXT SCORE"):
        w=Window(col-rad,row-rad,2*rad+1,2*rad+1)
        a=ps.read(1,window=w,boundless=True,fill_value=65535)
        v=a[a!=65535].astype(np.float32)/10000.0
        context_mean=float(v.mean()) if len(v) else 0.0
        context_high=float((v>=PROB_THRESHOLD).mean()) if len(v) else 0.0
        if context_mean < MIN_CONTEXT_MEAN: continue
        # Ranking rewards both a strong peak and broad local support.
        score=0.40*peak + 0.45*context_mean + 0.15*context_high
        coherent.append((score,peak,context_mean,context_high,row,col,x,y))

print("Coherent candidates before spatial NMS:",len(coherent))

# Pass 3: exact 10-m NMS using a spatial hash, highest coherent score first.
coherent.sort(key=lambda z:z[0],reverse=True)
accepted=[]; occupied={}
for rec in tqdm(coherent,desc="CANDIDATE NMS"):
    score,peak,cmean,chigh,row,col,x,y=rec
    kx=int(math.floor(x/grid_m)); ky=int(math.floor(y/grid_m)); ok=True
    for dx in (-1,0,1):
        for dy in (-1,0,1):
            for qx,qy in occupied.get((kx+dx,ky+dy),[]):
                if (x-qx)**2+(y-qy)**2 < PEAK_MIN_DISTANCE_M**2:
                    ok=False; break
            if not ok: break
        if not ok: break
    if ok:
        accepted.append(rec)
        occupied.setdefault((kx,ky),[]).append((x,y))

cand=pd.DataFrame(accepted,columns=["candidate_score","peak_prob","context_mean","context_high_frac",
                                    "row","col","x","y"])
cand=cand.sort_values("candidate_score",ascending=False).reset_index(drop=True)
n_before=len(cand)
if MAX_CANDIDATES is not None: cand=cand.head(MAX_CANDIDATES).copy()
cand["candidate_id"]=np.arange(len(cand),dtype=int)
cg=gpd.GeoDataFrame(cand,geometry=gpd.points_from_xy(cand.x,cand.y),crs=pcrs)
cg.to_file(OUT/"Baego_UNet_candidates.gpkg",layer="candidates",driver="GPKG")
print("Peak threshold:",PROB_THRESHOLD)
print("Minimum context mean:",MIN_CONTEXT_MEAN)
print("Minimum separation m:",PEAK_MIN_DISTANCE_M)
print("Candidates after NMS:",n_before)
print("Candidates retained for SAM3:",len(cg))
display(cg[["candidate_id","candidate_score","peak_prob","context_mean","context_high_frac"]].head(10))

# Prepare one compact RGB chip per retained candidate only.
manifest=[]
with rasterio.open(RASTER) as src:
    if src.crs!=pcrs: raise ValueError("RGB and probability raster CRS differ")
    px=float((abs(src.transform.a)+abs(src.transform.e))/2)
    chip_px=max(128,int(round(CHIP_SIZE_M/px)))
    if chip_px%2: chip_px+=1
    for r in tqdm(cg.itertuples(),total=len(cg),desc="RGB CHIPS"):
        row,col=src.index(r.x,r.y)
        w=Window(col-chip_px//2,row-chip_px//2,chip_px,chip_px)
        op=CHIPS/f"cand_{r.candidate_id:05d}.jpg"
        if FORCE_CHIPS or not op.exists():
            rgb=np.moveaxis(src.read([1,2,3],window=w,boundless=True,fill_value=0),0,-1).astype(np.uint8)
            Image.fromarray(rgb).save(op,quality=95)
        manifest.append(dict(candidate_id=r.candidate_id,path=str(op),x=r.x,y=r.y,
                             candidate_score=r.candidate_score,peak_prob=r.peak_prob,
                             context_mean=r.context_mean,context_high_frac=r.context_high_frac,
                             row_off=int(w.row_off),col_off=int(w.col_off),
                             height=chip_px,width=chip_px,gsd=px))
manifest=pd.DataFrame(manifest)
manifest.to_csv(OUT/"candidate_manifest.csv",index=False)
print("RGB chips:",len(manifest)," chip pixels:",chip_px)


## 2 — Isolated official SAM3 environment

In [ ]:
# SAM3 needs numpy<2, while current Colab geospatial packages need numpy>=2.
# Use uv to create a real isolated Python 3.12 environment; do NOT modify host Python.
UV=Path("/content/uv")
SAMENV=Path("/content/sam3env")
SAMREPO=Path("/content/sam3")

if not UV.exists():
    subprocess.run(["bash","-lc","curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/content sh"],check=True)
    # uv installer may name the binary directly under /content.
    candidates=[Path("/content/uv"),Path("/content/uv/uv"),Path.home()/".local/bin/uv"]
    UV=next((p for p in candidates if p.exists() and p.is_file()),None)
    assert UV is not None,"uv installation failed"
else:
    UV=Path("/content/uv")
print("uv:",subprocess.check_output([str(UV),"--version"],text=True).strip())

# Recreate only the isolated SAM environment if a previous failed venv is present.
if SAMENV.exists() and not (SAMENV/"bin/python").exists():
    shutil.rmtree(SAMENV)
if not (SAMENV/"bin/python").exists():
    subprocess.run([str(UV),"venv","--python","3.12",str(SAMENV)],check=True)

SAMPY=str(SAMENV/"bin/python")
# Reuse Colab's CUDA-enabled torch wheel via explicit install in the isolated env.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "torch==2.10.0","torchvision",
                "--index-url","https://download.pytorch.org/whl/cu128"],check=True)

if not SAMREPO.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/facebookresearch/sam3.git",str(SAMREPO)],check=True)
else:
    subprocess.run(["git","-C",str(SAMREPO),"pull","--ff-only"],check=True)

# Official SAM3 currently requires numpy>=1.26,<2. Keep that constraint inside this env only.
subprocess.run([str(UV),"pip","install","--python",SAMPY,"setuptools<82"],check=True)
# The image builder imports modules that use packages listed by Meta under
# notebook/dev extras (not only the minimal core dependencies). Install the
# official extras rather than discovering them one-by-one at runtime.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "-e",str(SAMREPO)+"[notebooks,dev]"],check=True)

subprocess.run([SAMPY,"-c",
    "import sys,numpy,torch,sam3; print('SAM Python',sys.version.split()[0],'numpy',numpy.__version__,'torch',torch.__version__,'CUDA',torch.cuda.is_available())"],
    check=True)
print("HOST unchanged: Python",sys.version.split()[0],"numpy",np.__version__,"rasterio",rasterio.__version__)


## 3 — Hugging Face authentication

In [ ]:
from google.colab import userdata
try: HF_TOKEN=userdata.get("HF_TOKEN")
except Exception: HF_TOKEN=None
assert HF_TOKEN,"Add your Hugging Face read token to Colab Secrets as HF_TOKEN."
print("HF_TOKEN found (value not displayed).")


## 4 — SAM3 delineation of U-Net candidates

SAM3 is now used as a **local delineator**, not a wall-to-wall detector. Each U-Net candidate supplies a small approximate box centred on the probability peak. SAM3 receives the local RGB chip plus that geometric prompt and returns a crown mask.

This is the computational advantage of the revised workflow: SAM3 is run only where the U-Net already provides evidence for Musanga.


In [ ]:
RUNNER=Path("/content/run_sam3_kvo06.py")
RUNNER.write_text(r'''import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); box_m=float(sys.argv[3]); force=sys.argv[4]=="1"
outdir.mkdir(parents=True,exist_ok=True); df=pd.read_csv(manifest)
if not torch.cuda.is_available(): raise RuntimeError("KVO_06 requires CUDA")
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__(); torch.inference_mode().__enter__()
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model=build_sam3_image_model(bpe_path=bpe); processor=Sam3Processor(model)
print("SAM3 ready; candidates=",len(df),flush=True)
for r in tqdm(df.itertuples(),total=len(df),desc="DELINEATE"):
    op=outdir/f"cand_{int(r.candidate_id):05d}.npz"
    if op.exists() and not force: continue
    img=Image.open(r.path).convert("RGB")
    state=processor.set_image(img)
    # Normalized centre-x, centre-y, width, height; candidate is chip centre.
    frac=min(0.95,box_m/(r.width*r.gsd))
    box=np.array([[0.5,0.5,frac,frac]],dtype=np.float32)
    out=processor.add_geometric_prompt(box=box,label=np.array([1],dtype=np.int32),state=state)
    masks=out["masks"].detach().cpu().numpy().astype(np.uint8)
    if masks.ndim==4: masks=masks[:,0]
    scores=out["scores"].detach().float().cpu().numpy()
    np.savez_compressed(op,masks=masks,scores=scores)
print("DONE",flush=True)
''')
env=os.environ.copy(); env["HF_TOKEN"]=HF_TOKEN
cmd=[SAMPY,"-u",str(RUNNER),str(OUT/"candidate_manifest.csv"),str(RAW),str(BOX_SIZE_M),"1" if FORCE_SAM else "0"]
print("Starting targeted SAM3 delineation with live output...")
p=subprocess.run(cmd,env=env)
print("Return code:",p.returncode)
if p.returncode!=0: raise RuntimeError("SAM3 delineation subprocess failed; error printed above.")


## 5 — Convert SAM3 masks to georeferenced candidate crown polygons

For each candidate we retain the SAM3 mask component containing the U-Net peak (the chip centre), convert it to map coordinates, and attach the original U-Net peak probability. Implausibly small/large objects are discarded.


In [ ]:
pred=[]
with rasterio.open(RASTER) as src:
    for r in tqdm(manifest.itertuples(),total=len(manifest),desc="POLYGONIZE"):
        z=np.load(RAW/f"cand_{int(r.candidate_id):05d}.npz")
        masks=z["masks"]; scores=z["scores"]
        if len(masks)==0: continue
        # Prefer highest-score mask containing chip centre.
        order=np.argsort(scores)[::-1]; chosen=None
        for j in order:
            m=masks[j].astype(np.uint8)
            if m.shape!=(r.height,r.width):
                m=np.array(Image.fromarray(m).resize((r.width,r.height),resample=Image.Resampling.NEAREST))
            if m[r.height//2,r.width//2]>0: chosen=(m,float(scores[j])); break
        if chosen is None: continue
        m,score=chosen
        wt=rasterio.windows.transform(Window(r.col_off,r.row_off,r.width,r.height),src.transform)
        geoms=[]
        for gj,val in shapes(m,mask=m.astype(bool),transform=wt):
            if val==1: geoms.append(shape(gj))
        if not geoms: continue
        pt=Point(r.x,r.y)
        geom=max((q for q in geoms if q.contains(pt) or q.touches(pt)),key=lambda q:q.area,default=None)
        if geom is None: continue
        area=geom.area
        if not (MIN_PRED_AREA_M2<=area<=MAX_PRED_AREA_M2): continue
        pred.append(dict(candidate_id=int(r.candidate_id),peak_prob=float(r.peak_prob),
                         sam_score=score,crown_area_m2=area,geometry=geom))
crowns=gpd.GeoDataFrame(pred,crs=src.crs)
crowns.to_file(OUT/"Baego_UNet_guided_SAM3_crowns.gpkg",layer="crowns",driver="GPKG")
print("Candidates:",len(manifest))
print("Valid SAM3 crown objects:",len(crowns))
display(crowns.drop(columns="geometry").head())


## 6 — Add object-level U-Net probability summaries

The peak probability detects the candidate; here we also summarize the KVO_04 probability raster inside each SAM3 crown. These are useful object-level attributes for later filtering and satellite aggregation.


In [ ]:
from rasterio.mask import mask as rio_mask
rows=[]
with rasterio.open(PROB) as ps:
    for r in tqdm(crowns.itertuples(),total=len(crowns),desc="OBJECT PROBABILITY"):
        arr,_=rio_mask(ps,[r.geometry],crop=True,filled=False,indexes=1)
        vals=np.asarray(arr.compressed(),dtype=np.float32)
        vals=vals[vals!=65535]/10000.0
        rows.append(dict(candidate_id=r.candidate_id,
                         mean_prob=float(np.mean(vals)) if len(vals) else np.nan,
                         median_prob=float(np.median(vals)) if len(vals) else np.nan,
                         p90_prob=float(np.quantile(vals,.9)) if len(vals) else np.nan))
stats=pd.DataFrame(rows)
crowns=crowns.merge(stats,on="candidate_id",how="left")
crowns.to_file(OUT/"Baego_UNet_guided_SAM3_crowns.gpkg",layer="crowns",driver="GPKG")
print(crowns[["peak_prob","mean_prob","median_prob","crown_area_m2"]].describe())


## 7 — Optional geometric QA against Crown_Nadine

`Crown_Nadine` is non-exhaustive, so it cannot provide whole-area precision. Where an automatically generated candidate crown overlaps a Nadine reference, however, we can report geometric agreement as a secondary QA. KVO_05 remains the authoritative dedicated boundary benchmark.


In [ ]:
def geom_iou(a,b):
    inter=a.intersection(b).area
    if inter<=0:return 0.0
    u=a.area+b.area-inter
    return inter/u if u>0 else 0.0

qa=[]
if ANNOT.exists() and len(crowns):
    ref=gpd.read_file(ANNOT)
    if ref.crs!=crowns.crs: ref=ref.to_crs(crowns.crs)
    ref=ref[ref.geometry.notna() & ~ref.geometry.is_empty].copy()
    ref=ref[ref.geometry.area>=5.0].copy()
    sidx=ref.sindex
    for r in crowns.itertuples():
        ids=list(sidx.query(r.geometry,predicate="intersects"))
        best=max((geom_iou(r.geometry,ref.iloc[i].geometry) for i in ids),default=0.0)
        qa.append((r.candidate_id,best))
qa=pd.DataFrame(qa,columns=["candidate_id","best_nadine_iou"])
if len(qa):
    qa.to_csv(OUT/"nadine_overlap_QA.csv",index=False)
    print("Candidate crowns overlapping Nadine at IoU >=0.25:",int((qa.best_nadine_iou>=.25).sum()))
    print("Candidate crowns overlapping Nadine at IoU >=0.50:",int((qa.best_nadine_iou>=.50).sum()))
    print("Median IoU among >=0.25 matches:",qa.loc[qa.best_nadine_iou>=.25,"best_nadine_iou"].median())
else: print("No Nadine overlap QA available.")


## 8 — Diagnostics and production summary

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
if len(crowns):
    ax.scatter(crowns.peak_prob,crowns.mean_prob,s=12,alpha=.6)
ax.set_xlabel("U-Net candidate peak probability"); ax.set_ylabel("Mean U-Net probability inside SAM3 crown")
ax.set_title("Baego: detector → delineator consistency")
plt.show()

summary={
    "probability_threshold":PROB_THRESHOLD,
    "minimum_candidate_separation_m":PEAK_MIN_DISTANCE_M,
    "n_candidates":int(len(manifest)),
    "n_valid_sam3_crowns":int(len(crowns)),
    "median_crown_area_m2":float(crowns.crown_area_m2.median()) if len(crowns) else None,
    "median_object_mean_probability":float(crowns.mean_prob.median()) if len(crowns) else None,
}
(OUT/"summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


## 9 — Interpretation / next step

KVO_06 deliberately avoids generic wall-to-wall SAM3 discovery.

- **KVO_03:** independent leave-one-site-out evidence that the U-Net transfers between sites.
- **KVO_04:** final continuous Musanga probability maps.
- **KVO_05:** zero-shot evidence that SAM3 can delineate a crown accurately when approximately located.
- **KVO_06:** combines those strengths efficiently: U-Net detects likely Musanga locations and SAM3 converts selected candidates into individual crown objects.

The resulting object layer can later be aggregated to satellite support using crown count, crown area fraction, and object-level Musanga probabilities.
